# Практическая работа №3. EDA и Pandas
## Вариант 3. Продажи кофейни

**Датасет:** Coffee_Shop_Sales.xlsx  

В этой работе выполняется разведочный анализ данных (EDA) по продажам кофейни с использованием библиотеки Pandas.


## 1. Выполнение EDA датасета

### Загрузка данных

Сначала импортируем необходимые библиотеки и загружаем датасет с помощью `pd.read_excel()`.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

df = pd.read_excel('Coffee_Shop_Sales.xlsx')
df.head()


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


Датасет успешно загружен. В первых строках видны транзакции из локации Lower Manhattan: кофе, чай, горячий шоколад и выпечка.


### Размерность датасета

Атрибут `shape` возвращает кортеж (количество строк, количество столбцов).


In [2]:
print('Размерность датасета (строки, столбцы):', df.shape)


Размерность датасета (строки, столбцы): (149116, 11)


Датасет содержит **149 116** транзакций и **11** столбцов. 


### Информация о датасете

Метод `info()` показывает типы данных, количество ненулевых значений и использование памяти.


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    149116 non-null  int64         
 1   transaction_date  149116 non-null  datetime64[us]
 2   transaction_time  149116 non-null  object        
 3   transaction_qty   149116 non-null  int64         
 4   store_id          149116 non-null  int64         
 5   store_location    149116 non-null  str           
 6   product_id        149116 non-null  int64         
 7   unit_price        149116 non-null  float64       
 8   product_category  149116 non-null  str           
 9   product_type      149116 non-null  str           
 10  product_detail    149116 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(4), object(1), str(4)
memory usage: 19.8+ MB


Все столбцы полностью заполнены (149116 non-null). Пропусков нет. Типы данных корректны: числовые, дата и строки.


### Первичные статистические характеристики

Метод `describe()` выводит основные статистики только по числовым столбцам.


In [4]:
df.describe()


,transaction_id,transaction_date,transaction_qty,store_id,product_id,unit_price
count,149116.000000,149116,149116.000000,149116.000000,149116.000000,149116.000000
mean,74737.371872,2023-04-15 11:50:32.173609,1.438276,5.342063,47.918607,3.382219
min,1.000000,2023-01-01 00:00:00,1.000000,3.000000,1.000000,0.800000
25%,37335.750000,2023-03-06 00:00:00,1.000000,3.000000,33.000000,2.500000
50%,74727.500000,2023-04-24 00:00:00,1.000000,5.000000,47.000000,3.000000
75%,112094.250000,2023-05-30 00:00:00,2.000000,8.000000,60.000000,3.750000
max,149456.000000,2023-06-30 00:00:00,8.000000,8.000000,87.000000,45.000000
std,43153.600016,NaN,0.542509,2.074241,17.930020,2.658723


Для **unit_price**: среднее ≈ 3,38 $, медиана = 3,00 $, минимум = 0,80 $, максимум = 45,00 $.  
Для **transaction_qty**: среднее ≈ 1,44, медиана = 1, максимум = 8.  
Большой разброс цен связан с наличием дорогих товаров (кофейные зёрна).


### Количество уникальных значений

Метод `nunique()` считает число уникальных значений в каждом столбце.


In [5]:
df.nunique()


transaction_id      149116
transaction_date       181
transaction_time     25762
transaction_qty          6
store_id                 3
store_location           3
product_id              80
unit_price              41
product_category         9
product_type            29
product_detail          80
dtype: int64

Все `transaction_id` уникальны (149116). Есть 3 локации, 9 категорий продуктов, 80 различных товаров, 41 уникальная цена.


### Описание колонок датасета

- **transaction_id** — уникальный идентификатор транзакции  
- **transaction_date** — дата транзакции  
- **transaction_time** — время транзакции  
- **transaction_qty** — количество единиц товара в транзакции  
- **store_id** — идентификатор магазина (3, 5, 8)  
- **store_location** — локация магазина (Astoria, Hell's Kitchen, Lower Manhattan)  
- **product_id** — идентификатор продукта  
- **unit_price** — цена за единицу товара (USD)  
- **product_category** — категория продукта (Coffee, Tea, Bakery и др.)  
- **product_type** — тип продукта (Gourmet brewed coffee, Scone и т.д.)  
- **product_detail** — детальное название продукта


### Мода, среднее и медиана по unit_price

В качестве значимой числовой колонки выбран **unit_price** — цена напрямую влияет на выручку.


In [6]:
print('Мода unit_price:', df['unit_price'].mode().tolist())
print('Среднее unit_price:', round(df['unit_price'].mean(), 4))
print('Медиана unit_price:', df['unit_price'].median())


Мода unit_price: [3.0]
Среднее unit_price: 3.3822
Медиана unit_price: 3.0


Мода и медиана равны 3,0 $, среднее выше (3,38 $). Это типичный признак правоскошенного распределения: редкие дорогие товары «тянут» среднее вверх.


### Уникальные значения категориальных столбцов

Метод `unique()` показывает все различные значения столбца.


In [19]:
print('store_location:', df['store_location'].unique().tolist())
print()
print('product_category:', df['product_category'].unique().tolist())
print()
print('transaction_qty:', sorted(df['transaction_qty'].unique().tolist()))
print()
print('product_type:')
print(df['product_type'].unique().tolist())


store_location: ['Lower Manhattan', "Hell's Kitchen", 'Astoria']

product_category: ['Coffee', 'Tea', 'Drinking Chocolate', 'Bakery', 'Flavours', 'Loose Tea', 'Coffee beans', 'Packaged Chocolate', 'Branded']

transaction_qty: [1, 2, 3, 4, 6, 8]

product_type:
['Gourmet brewed coffee', 'Brewed Chai tea', 'Hot chocolate', 'Drip coffee', 'Scone', 'Barista Espresso', 'Brewed Black tea', 'Brewed Green tea', 'Brewed herbal tea', 'Biscotti', 'Pastry', 'Organic brewed coffee', 'Premium brewed coffee', 'Regular syrup', 'Herbal tea', 'Gourmet Beans', 'Organic Beans', 'Sugar free syrup', 'Drinking Chocolate', 'Premium Beans', 'Chai tea', 'Green beans', 'Espresso Beans', 'Green tea', 'Organic Chocolate', 'Housewares', 'Black tea', 'House blend Beans', 'Clothing']


Три локации, девять категорий, количество товара принимает значения 1, 2, 3, 4, 6 или 8. 29 типов продуктов.


### Пропущенные значения

Процент пропусков вычисляется через `isnull().mean() * 100`.


In [8]:
print((df.isnull().mean() * 100).round(2))


transaction_id      0.0
transaction_date    0.0
transaction_time    0.0
transaction_qty     0.0
store_id            0.0
store_location      0.0
product_id          0.0
unit_price          0.0
product_category    0.0
product_type        0.0
product_detail      0.0
dtype: float64


Пропусков нет (0 % по всем столбцам). Очистка NaN не требуется.  

Строка «no data» не подходит для числовых столбцов: она нарушает тип данных (float/int) и делает невозможными операции mean, sum, skew и т.д. Для категориальных столбцов при наличии пропусков использовали бы `fillna('no data')`.


### Дубликаты

Метод `duplicated().sum()` считает количество полных дубликатов.


In [9]:
print('Количество полных дубликатов:', df.duplicated().sum())
print('Размерность:', df.shape)


Количество полных дубликатов: 0
Размерность: (149116, 11)


Дубликатов нет. Каждая строка — уникальная транзакция. Удаление не требуется, смысл датасета сохранён.


## 2. Коэффициент эксцесса, асимметрии, CV

Для расчёта выбран столбец **unit_price** — непрерывный числовой признак с широким диапазоном и бизнес-значимостью.

### 2.1–2.2. Асимметрия и эксцесс


In [20]:
col = 'unit_price'
stats = pd.DataFrame({
    'Показатель': ['Среднее', 'Медиана', 'Стд. отклонение', 'Skew (асимметрия)', 'Kurtosis (эксцесс)'],
    'Значение': [
        round(df[col].mean(), 4),
        df[col].median(),
        round(df[col].std(), 4),
        round(skew(df[col]), 4),
        round(kurtosis(df[col]), 4)
    ]
})
stats


,Показатель,Значение
0,Среднее,3.3822
1,Медиана,3.0000
2,Стд. отклонение,2.6587
3,Skew (асимметрия),8.4550
4,Kurtosis (эксцесс),98.9161


**Интерпретация:**

- **Skew = 8,46**  — сильная асимметрия. Большая часть покупок имеет цену 2–4 , редкие дорогие товары (зёрна до 45 ) тянут среднее вправо.
- **Kurtosis = 98,9** (и — очень тяжёлые хвосты, **много выбросов** с большим значением. Распределение далеко от нормального.
- Среднее (3,38) > медианы (3,0) — подтверждает скошенность к болтшим величинам.


### 2.3. Коэффициент вариации (CV)


In [11]:
cv = df['unit_price'].std() / df['unit_price'].mean() * 100
print(f'CV (unit_price) = {cv:.2f}%')


CV (unit_price) = 78.61%


CV = 78,6 % — **высокая вариативность**. Цены сильно различаются (от 0,8 за сироп до 45 за зёрна). Это нормально для широкого ассортимента кофейни.


## 3. Фильтрация, срезы, группировка и сортировка

### Фильтрация через query()

Отберём транзакции из Lower Manhattan, где купили больше 2 единиц товара.


In [12]:
q1 = df.query("store_location == 'Lower Manhattan' and transaction_qty > 2")
print('Количество строк:', len(q1))
q1.head(3)


Количество строк: 3305


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
3842,3872,2023-01-08,07:37:12,3,5,Lower Manhattan,63,0.8,Flavours,Regular syrup,Carmel syrup
3977,4007,2023-01-08,09:34:26,3,5,Lower Manhattan,63,0.8,Flavours,Regular syrup,Carmel syrup
4088,4118,2023-01-08,10:59:55,3,5,Lower Manhattan,65,0.8,Flavours,Sugar free syrup,Sugar Free Vanilla syrup


Получено 3305 крупных заказов из Lower Manhattan.


### Сортировка через sort_values()

Отсортируем по unit_price по убыванию, чтобы увидеть самые дорогие позиции.


In [13]:
q2 = df.sort_values('unit_price', ascending=False)
q2[['transaction_id', 'product_detail', 'unit_price', 'store_location']].head(3)


,transaction_id,product_detail,unit_price,store_location
64745,64915,Civet Cat,45.0,Hell's Kitchen
125082,125419,Civet Cat,45.0,Lower Manhattan
136500,136837,Civet Cat,45.0,Astoria


В топе — кофейные зёрна Civet Cat по 45 $. ( кофе за 40 баксов...)



### Срезы данных

**1) Через loc** — выборка по именам столбцов:


In [14]:
df.loc[0:4, ['store_location', 'unit_price', 'product_category']]


,store_location,unit_price,product_category
0,Lower Manhattan,3.0,Coffee
1,Lower Manhattan,3.1,Tea
2,Lower Manhattan,4.5,Drinking Chocolate
3,Lower Manhattan,2.0,Coffee
4,Lower Manhattan,3.1,Tea


**2) Через loc с условием** — только товары категории Coffee:


In [15]:
df.loc[df['product_category'] == 'Coffee', ['product_detail', 'unit_price', 'transaction_qty']].head()


,product_detail,unit_price,transaction_qty
0,Ethiopia Rg,3.00,2
3,Our Old Time Diner Blend Sm,2.00,1
6,Our Old Time Diner Blend Sm,2.00,1
7,Columbian Medium Roast Sm,2.00,2
8,Latte Rg,4.25,1


**3) Через iloc** — позиционный срез (строки 0–4, столбцы 4–8):


In [16]:
df.iloc[0:5, 4:9]


,store_id,store_location,product_id,unit_price,product_category
0,5,Lower Manhattan,32,3.0,Coffee
1,5,Lower Manhattan,57,3.1,Tea
2,5,Lower Manhattan,59,4.5,Drinking Chocolate
3,5,Lower Manhattan,22,2.0,Coffee
4,5,Lower Manhattan,57,3.1,Tea


### Группировка

Сгруппируем по store_location и посчитаем сумму qty, среднюю цену и количество транзакций.


In [17]:
grouped = df.groupby('store_location').agg(
    qty_sum=('transaction_qty', 'sum'),
    price_mean=('unit_price', 'mean'),
    num_transactions=('transaction_id', 'count')
)
grouped


,qty_sum,price_mean,num_transactions
store_location,,,
Astoria,70991,3.384897,50599
Hell's Kitchen,71737,3.394218,50735
Lower Manhattan,71742,3.366643,47782


Три локации работают на сопоставимом уровне: около 48–51 тыс. транзакций, средняя цена ≈ 3,37–3,39 $.


## 4. Топ-5 записей

Выберем топ-5 транзакций по **выручке** (revenue = transaction_qty × unit_price).  
Этот признак наиболее значим для бизнеса — он показывает, какие продажи принесли максимальную денежную отдачу.


In [21]:
df['revenue'] = df['transaction_qty'] * df['unit_price']
top5 = df.sort_values('revenue', ascending=False).head(5)
top5 = top5.reset_index(drop=True)
top5


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,revenue
0,133674,2023-06-17,10:41:11,8,8,Hell's Kitchen,8,45.0,Coffee beans,Premium Beans,Civet Cat,360.0
1,149043,2023-06-30,11:18:31,8,8,Hell's Kitchen,8,45.0,Coffee beans,Premium Beans,Civet Cat,360.0
2,133744,2023-06-17,11:18:31,8,8,Hell's Kitchen,8,45.0,Coffee beans,Premium Beans,Civet Cat,360.0
3,69151,2023-04-17,11:18:31,8,8,Hell's Kitchen,8,45.0,Coffee beans,Premium Beans,Civet Cat,360.0
4,98233,2023-05-17,09:05:20,8,8,Hell's Kitchen,8,45.0,Coffee beans,Premium Beans,Civet Cat,360.0


Все пять записей — покупки **8 упаковок** кофейных зёрен **Civet Cat** по **45 **. Выручка каждой = **360 **.

На попадание в рейтинг повлияли:
1. Наличие в ассортименте дорогих весовых товаров (Coffee beans).
2. Возможность купить сразу несколько единиц (qty = 8).
3. Максимальная unit_price в датасете (45 $).


## Заключение

Датасет продаж кофейни 149 116 транзакций без пропусков и дубликатов. Основные продажи — категории Coffee и Tea. Распределение цен сильно правоскошено из-за дорогих товаров. Три локации работают примерно одинаково. Наибольшую выручку дают редкие крупные покупки дорогих позиций (кофейные зёрна).
